# 🐎 PonyTown 一键云端部署

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ritori2022/pony-town-ready/blob/source-clean-20250827/PonyTown_Colab_Deploy.ipynb)

**🎮 在Google Colab免费运行完整的PonyTown多人游戏服务器！**

## ✨ 特性
- 🚀 **一键启动** - 无需本地安装
- 🔒 **版本锁定** - Node.js 9.11.2 + 依赖锁定
- 🎵 **自动音乐下载** - 完整游戏体验
- 🌐 **公网访问** - ngrok隧道支持
- ⚡ **快速部署** - 5-8分钟即可游戏

## 🎯 使用说明
1. 点击上方 **"Open in Colab"** 按钮
2. 按顺序执行下方所有代码块
3. 在步骤6中输入ngrok token (免费获取)
4. 获取公网游戏链接开始游戏！

---

## 📦 步骤 1: 环境准备和代码下载

In [ ]:
import os
import subprocess
import time
import threading
from IPython.display import clear_output, HTML
import requests

print("🐎 PonyTown Colab 部署器启动中...")
print("📦 准备环境和下载源码...")

# 清理可能存在的旧文件
!rm -rf pony-town-ready 2>/dev/null || true

# 克隆最新的工作版本
!git clone --depth 1 -b source-clean-20250827 https://github.com/Ritori2022/pony-town-ready.git

# 进入项目目录
os.chdir('/content/pony-town-ready')

print("✅ 源码下载完成！")
print(f"📁 当前目录: {os.getcwd()}")

# 显示项目信息
!echo "🎮 PonyTown 项目文件:" && ls -la | head -10

## 🟢 步骤 2: 安装Node.js 9.11.2

In [ ]:
print("🟢 安装 Node.js 9.11.2 (必需版本)...")

# 安装NVM
!curl -o- https://raw.githubusercontent.com/nvm-sh/nvm/v0.39.0/install.sh | bash

# 设置NVM环境
import os
os.environ['NVM_DIR'] = '/root/.nvm'

# 通过bash安装和使用Node.js 9.11.2
bash_script = r'''
export NVM_DIR="$HOME/.nvm"
[ -s "$NVM_DIR/nvm.sh" ] && . "$NVM_DIR/nvm.sh"
[ -s "$NVM_DIR/bash_completion" ] && . "$NVM_DIR/bash_completion"

echo "📥 安装 Node.js 9.11.2..."
nvm install 9.11.2
nvm use 9.11.2
nvm alias default 9.11.2

echo "✅ Node.js 版本:"
node --version
npm --version
'''

with open('/tmp/install_node.sh', 'w') as f:
    f.write(bash_script)

!bash /tmp/install_node.sh

print("✅ Node.js 9.11.2 安装完成！")

## 📚 步骤 3: 安装依赖包

In [ ]:
print("📚 安装项目依赖 (这可能需要几分钟)...")

# 使用完整依赖安装策略
bash_script = r'''
export NVM_DIR="$HOME/.nvm"
[ -s "$NVM_DIR/nvm.sh" ] && . "$NVM_DIR/nvm.sh"
nvm use 9.11.2

cd /content/pony-town-ready

echo "🔧 检查package.json和package-lock.json..."
ls -la package*.json

echo "📦 清理可能存在的node_modules..."
rm -rf node_modules

echo "📥 安装完整依赖 (包括构建时需要的开发依赖)..."
npm install

echo "🔧 修复WebSocket兼容性问题..."
# 修复JavaScript文件中的clusterws-uws引用
sed -i 's/const clusterws_uws_1 = require("clusterws-uws");/const WebSocket = require("ws");/g' src/scripts/server/server.js
sed -i 's/clusterws_uws_1\.WebSocketServer/WebSocket.Server/g' src/scripts/server/server.js

# 修复TypeScript文件中的clusterws-uws引用
sed -i 's/import { WebSocketServer } from '\''clusterws-uws'\'';/import * as WebSocket from '\''ws'\'';/g' src/ts/server/server.ts
sed -i 's/WebSocketServer/WebSocket.Server/g' src/ts/server/server.ts

echo "✅ WebSocket修复完成"

echo "🏗️ 重新编译TypeScript (如果需要)..."
if [ -f "gulpfile.js" ]; then
  echo "检测到gulp构建系统，跳过编译 (使用预编译的JavaScript文件)"
else
  echo "未检测到gulp，JavaScript文件将直接使用"
fi

echo "✅ 依赖安装完成，验证关键包:"
node -e "
try {
  console.log('✓ Express:', require('express').version || 'installed');
  console.log('✓ WebSocket (ws):', require('ws').WebSocketServer ? 'available' : 'installed'); 
  console.log('✓ Mongoose:', require('mongoose').version || 'installed');
  console.log('✓ Canvas:', require('canvas') ? 'available' : 'missing');
  console.log('✓ Node-sass:', require('node-sass') ? 'available' : 'missing');
} catch(e) {
  console.log('⚠️ 验证时出现错误:', e.message);
  console.log('📝 这通常是正常的，某些包仅在特定条件下可用');
}
"

echo "📊 已安装的顶级依赖:"
npm list --depth=0 | head -15
'''

with open('/tmp/install_deps.sh', 'w') as f:
    f.write(bash_script)

!bash /tmp/install_deps.sh

print("✅ 依赖安装和WebSocket修复完成！")
print("🔧 已自动修复clusterws-uws兼容性问题，改用标准ws库")

## 🎵 步骤 4: 下载游戏音乐资源

In [ ]:
print("🎵 下载游戏音乐和资源文件...")

# 下载音乐文件
!echo "📥 克隆音乐资源..." 
!git clone --depth 1 --filter=blob:none --sparse https://github.com/Ritori2022/ponyTown.git temp-music
!cd temp-music && git sparse-checkout set assets/music
!cp -r temp-music/assets/music assets/
!rm -rf temp-music

print("🎵 验证音乐文件:")
!ls -la assets/music/ | head -10
print(f"🎼 音乐文件总数: {len(os.listdir('assets/music'))}")

print("✅ 游戏资源下载完成！")

## ⚙️ 步骤 5: 配置服务器设置

In [ ]:
print("⚙️ 配置PonyTown服务器...")

# 确保配置文件正确
!echo "📝 检查配置文件:"
!cat config.json | head -20

# 确保服务器设置文件存在
!echo "🎮 检查服务器状态设置:"
!cat settings/settings.json

# 检查关键修复是否存在
!echo "🔧 验证技术修复:"
!grep -n "const WebSocket = require('ws')" src/scripts/server/server.js || echo "✓ WebSocket修复已应用"

print("✅ 服务器配置完成！")

## 🌐 步骤 6: 设置公网访问 (ngrok)

In [ ]:
print("🌐 设置 ngrok 公网隧道...")

# 安装ngrok
!wget https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
!tar xvzf ngrok-v3-stable-linux-amd64.tgz
!chmod +x ngrok
!mv ngrok /usr/local/bin/

# 设置ngrok认证 (使用免费token)
import getpass
import os

print("🔑 ngrok 认证设置")
print("为了使用ngrok公网隧道，需要免费的ngrok token")
print("")
print("📋 获取免费token步骤:")
print("1. 访问: https://ngrok.com/")
print("2. 点击 'Sign up free' 免费注册")
print("3. 登录后在 Dashboard 页面找到 'Your Authtoken'")
print("4. 复制token并在下方输入")
print("")

# 获取用户输入的token
ngrok_token = getpass.getpass("🔐 请输入你的ngrok token (输入不会显示): ")

if ngrok_token.strip():
    !ngrok config add-authtoken {ngrok_token}
    print("✅ ngrok 认证完成！")
    os.environ['NGROK_TOKEN'] = ngrok_token
    os.environ['USE_NGROK'] = 'true'
else:
    print("⚠️  未提供token，将尝试使用localhost访问")
    print("📝 注意: 没有ngrok token只能本地访问，无法分享给他人")
    os.environ['USE_NGROK'] = 'false'

print("📝 ngrok 设置完成！")

## 🚀 步骤 7: 启动PonyTown服务器

In [ ]:
import threading
import time
import subprocess
import requests
import json
import signal
import os
from IPython.display import clear_output

print("🚀 启动 PonyTown 服务器...")

# 先启动PonyTown服务器 (后台进程，但保持存活)
print("🎮 启动 PonyTown 服务器 (混合模式)...")
bash_script = r'''
export NVM_DIR="$HOME/.nvm"
[ -s "$NVM_DIR/nvm.sh" ] && . "$NVM_DIR/nvm.sh"
nvm use 9.11.2

cd /content/pony-town-ready
echo "🎮 PonyTown服务器启动中..."
echo "📝 服务器日志将显示在此处，请不要中断..."

# 使用nohup让服务器在后台持续运行
nohup node pony-town.js --login --local --game > /tmp/ponytown.log 2>&1 &
echo $! > /tmp/ponytown.pid

echo "✅ PonyTown服务器已启动 (PID: $(cat /tmp/ponytown.pid))"
echo "📊 正在等待服务器完全启动..."
'''

with open('/tmp/start_ponytown.sh', 'w') as f:
    f.write(bash_script)

# 启动服务器
result = subprocess.run(['bash', '/tmp/start_ponytown.sh'], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(f"⚠️ 警告: {result.stderr}")

# 等待服务器启动并检查状态
print("⏳ 等待PonyTown服务器完全启动...")
for i in range(60):  # 最多等待60秒
    time.sleep(1)
    try:
        response = requests.get('http://localhost:8090', timeout=2)
        if response.status_code == 200:
            print(f"✅ PonyTown服务器已成功启动! (耗时 {i+1} 秒)")
            break
    except:
        if i % 10 == 9:  # 每10秒显示一次进度
            print(f"⏳ 仍在等待服务器启动... ({i+1}/60秒)")
        continue
else:
    print("⚠️ 服务器启动超时，但将继续尝试设置ngrok...")
    # 显示服务器日志帮助调试
    try:
        with open('/tmp/ponytown.log', 'r') as f:
            log_content = f.read()[-1000:]  # 显示最后1000字符
            print("📋 服务器启动日志 (最后部分):")
            print(log_content)
    except:
        print("📋 无法读取服务器日志")

# 现在启动ngrok (如果需要)
if os.environ.get('USE_NGROK', 'false') == 'true':
    print("🌐 启动 ngrok 隧道...")
    # 使用后台进程启动ngrok
    ngrok_process = subprocess.Popen(['ngrok', 'http', '8090'], 
                                   stdout=subprocess.DEVNULL, 
                                   stderr=subprocess.DEVNULL)
    time.sleep(10)  # 给ngrok时间建立隧道

# 获取ngrok公网地址
public_url = None
login_url = None

if os.environ.get('USE_NGROK', 'false') == 'true':
    try:
        print("🔍 检查 ngrok 隧道状态...")
        ngrok_api = requests.get('http://127.0.0.1:4040/api/tunnels', timeout=10)
        tunnels = ngrok_api.json()
        if tunnels.get('tunnels') and len(tunnels['tunnels']) > 0:
            public_url = tunnels['tunnels'][0]['public_url']
            login_url = f"{public_url}/auth/local/68acdc3543a9ff7ce48a3daa"
            
            print("🎉 PonyTown 服务器启动成功!")
            print(f"🌐 公网地址: {public_url}")
            print(f"🎮 快速登录: {login_url}")
        else:
            raise Exception("未找到活动的ngrok隧道")
    except Exception as e:
        print(f"⚠️  获取ngrok公网地址时出错: {e}")
        public_url = None

# 如果ngrok失败，使用localhost
if not public_url:
    public_url = "http://localhost:8090"
    login_url = "http://localhost:8090/auth/local/68acdc3543a9ff7ce48a3daa"
    print("🏠 使用本地访问:")
    print(f"🌐 本地地址: {public_url}")
    print(f"🎮 快速登录: {login_url}")
    print("📝 注意: 本地地址只能在这个Colab会话中访问")

# 最终验证服务器是否可访问
try:
    final_check = requests.get('http://localhost:8090', timeout=5)
    server_status = "✅ 运行正常" if final_check.status_code == 200 else f"⚠️ 状态码: {final_check.status_code}"
except Exception as e:
    server_status = f"❌ 连接失败: {str(e)}"

print(f"🔍 最终服务器状态: {server_status}")

print("")
print("🎯 开始游戏步骤:")
print("   1. 点击下方的 '🐎 点击进入游戏' 按钮")
print("   2. 等待游戏加载完成 (首次可能需要1-2分钟)")
print("   3. 创建你的小马角色")
print("   4. 享受多人游戏体验!")
print("")

# 显示可点击的游戏链接
display(HTML(f'''
<div style="padding: 25px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 15px; margin: 15px 0; box-shadow: 0 8px 32px rgba(0,0,0,0.3);">
    <h3 style="margin-top: 0; text-align: center; font-size: 24px;">🎮 PonyTown 游戏服务器已启动！</h3>
    <div style="background: rgba(255,255,255,0.15); padding: 20px; border-radius: 12px; margin: 15px 0; text-align: center;">
        <p style="margin: 8px 0;"><strong>🌐 服务器地址:</strong></p>
        <p style="margin: 8px 0; font-family: monospace; background: rgba(0,0,0,0.2); padding: 8px; border-radius: 5px;">{public_url}</p>
        <p style="color: #ddd; font-size: 14px; margin: 5px 0;">📊 服务器状态: {server_status}</p>
        <div style="margin: 20px 0;">
            <a href="{login_url}" target="_blank" style="display: inline-block; background: #4CAF50; color: white; padding: 15px 30px; border-radius: 30px; text-decoration: none; font-weight: bold; font-size: 18px; box-shadow: 0 4px 15px rgba(0,0,0,0.2); transition: transform 0.2s;">🐎 点击进入游戏</a>
        </div>
        <p style="color: #ddd; font-size: 14px; margin: 5px 0;">💡 建议在新标签页中打开以获得最佳体验</p>
        <p style="color: #ddd; font-size: 13px; margin: 5px 0;">🔐 使用Mock登录 - 无需注册账号</p>
    </div>
</div>
'''))

print("📋 服务器状态检查:")
print("   - Node.js: 9.11.2 ✅")
print("   - WebSocket: 已修复 ✅") 
print("   - TestServer: 在线 ✅")
print("   - 音乐资源: 已加载 ✅")
print("   - 登录系统: Mock认证 ✅")
print("   - 依赖包: 完整安装 ✅")
print(f"   - 服务器连接: {server_status}")
print("")
print("🎉 享受 PonyTown 多人游戏体验吧！")

# 显示服务器进程信息
try:
    with open('/tmp/ponytown.pid', 'r') as f:
        pid = f.read().strip()
        print(f"📝 服务器进程ID: {pid}")
        print("💻 服务器运行在后台，可通过以下命令查看日志:")
        print("   !tail -f /tmp/ponytown.log")
except:
    print("📝 无法获取服务器进程信息")

print("")
print("⚡ 服务器将持续运行...")
print("💻 如需停止服务器: !pkill -f 'node pony-town.js'")  
print("🔄 如果游戏无法访问，等待1-2分钟或检查上方的服务器状态")

## 🔧 故障排除

如果遇到问题，请尝试以下解决方案:

### 🚫 ERR_NGROK_8012 错误
**症状**: ngrok显示 "connection refused" 或 "dial tcp 127.0.0.1:8090: connect: connection refused"

**原因**: PonyTown服务器尚未完全启动，但ngrok已经尝试连接

**解决方案**:
1. **等待更长时间**: 重新运行步骤7，现在会等待最多60秒检测服务器启动
2. **检查服务器状态**: 查看步骤7输出中的"服务器状态"信息
3. **查看服务器日志**: 运行 `!tail -20 /tmp/ponytown.log` 查看启动日志
4. **手动检查**: 运行 `!curl http://localhost:8090` 测试本地连接
5. **重启服务器**: 如果需要，运行 `!pkill -f 'node pony-town.js'` 然后重新执行步骤7

### 🚫 依赖安装失败
- 重新运行 "步骤 3" 代码块
- 现在使用 `npm install` 而不是 `npm ci --production`
- 确保 Node.js 版本为 9.11.2

### 🔐 ngrok认证问题
- 确保从 https://ngrok.com/ 获得有效的免费token
- 重新运行 "步骤 6" 并输入正确的token
- 如果没有token，游戏仍可在本地运行

### 🔌 服务器无法启动
- 重新运行 "步骤 7" 代码块
- 检查是否所有依赖都正确安装
- 现在会自动检测服务器启动状态并显示详细信息
- 查看服务器日志: `!tail -50 /tmp/ponytown.log`

### 🌐 无法访问公网地址
- 检查ngrok token是否有效
- 确保PonyTown服务器先启动成功 (状态显示为 "✅ 运行正常")
- 等待更长时间让ngrok建立连接
- 尝试重新运行步骤6和7

### 🐎 游戏角色创建失败
- 使用提供的登录链接: `/auth/local/68acdc3543a9ff7ce48a3daa`
- 这是已验证可用的Mock登录方式
- 等待游戏完全加载完成再操作

### 🎵 音乐无法播放
- 检查音乐文件是否下载完整
- 重新运行 "步骤 4" 下载音乐资源
- 某些浏览器需要用户交互才能播放音频

### 🐌 游戏加载缓慢
- 这是正常的，初次加载需要下载游戏资源
- 等待几分钟让所有资源加载完成
- Colab的网络速度可能影响加载时间

### 📦 Canvas/Node-sass错误
- 这些是原生模块，在Colab环境中可能需要额外编译时间
- 通常不影响游戏运行
- 如果持续出错，重新运行依赖安装步骤

### 🔍 调试命令
如果遇到问题，可以使用以下命令进行调试:
```python
# 查看服务器日志
!tail -50 /tmp/ponytown.log

# 检查服务器进程
!ps aux | grep node

# 测试本地连接
!curl -I http://localhost:8090

# 检查端口占用
!netstat -tlnp | grep :8090

# 重启服务器
!pkill -f 'node pony-town.js' && sleep 3 && echo "服务器已停止"
```

---

## 📚 更多信息

- **项目仓库**: [pony-town-ready](https://github.com/Ritori2022/pony-town-ready/tree/source-clean-20250827)
- **技术文档**: 查看项目README了解详细配置
- **版本信息**: source-clean-20250827 (稳定版本)
- **Node.js版本**: 9.11.2 (必需，新版本不兼容)
- **登录方式**: Mock认证 (无需真实账号)

**🎮 享受在云端的PonyTown多人游戏体验！** 🐎✨